# Knowledge Hub · Chatbot
Caso de estudio Softtek

El chatbot utilizará el buscador multilingüe del prototipo para consultar el corpus y citar documentos. La primera prueba verifica que podemos recuperar el identificador, el título y el texto que respaldarán una respuesta.

### 1. Preparación del entorno

Descargamos la versión del prototipo verificada al cierre del punto 1 e instalamos las dependencias de búsqueda en CPU.

In [ ]:
%%bash
set -euo pipefail

repo=/content/knowledge-hub-chatbot-base
version=7c2f7d3
if [ ! -d "$repo/.git" ]; then
    git clone --quiet --branch main https://github.com/ierisaac2005-png/softtek-knowledge-hub.git "$repo"
fi
cd "$repo"
git switch --quiet --detach "$version"
python -m venv --without-pip .venv
python -m pip --python .venv/bin/python install -q --index-url https://download.pytorch.org/whl/cpu torch
python -m pip --python .venv/bin/python install -q -r requirements-search.txt
printf 'Versión del prototipo: '
git rev-parse --short HEAD

Versión del prototipo: 7c2f7d3


### 2. Recuperación de evidencia

La pregunta de control proviene de la evaluación del prototipo. El documento esperado es IR07. Mostramos los tres primeros resultados y el contenido del primero para comprobar si explica la causa del retraso.

In [ ]:
%%bash
set -euo pipefail
cd /content/knowledge-hub-chatbot-base

PYTHONPATH=src .venv/bin/python - <<'PYTHON'
from kms_utils import load_documents
from semantic_search import MultilingualSearch

pregunta = "¿Por qué los pedidos se procesaron con tres horas de retraso?"
buscador = MultilingualSearch(load_documents())
resultados = buscador.search(pregunta, top_k=3)

print("Pregunta:", pregunta)
print(resultados[["id", "title", "similarity"]].to_string(index=False))
primero = resultados.iloc[0]
print(f"\nFuente recuperada: [{primero['id']}] {primero['title']}")
print(primero["text"])
assert primero["id"] == "IR07", "Revisar el ranking: se esperaba IR07 en primer lugar."
PYTHON

Pregunta: ¿Por qué los pedidos se procesaron con tres horas de retraso?
  id                                             title  similarity
IR07                Post-mortem: Message Queue Backlog    0.801195
IR11 Incident Report: Third-Party Email Service Outage    0.799526
IR01                  Post-mortem: Payments API Outage    0.790485

Fuente recuperada: [IR07] Post-mortem: Message Queue Backlog
Order events accumulated in the queue and were processed with a delay of three hours. The root cause was a consumer that crashed due to an unhandled malformed message. A fix was deployed and monitoring of queue depth was added to the alerts.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4266.71it/s]


### Primera respuesta del chatbot con una fuente recuperada

El buscador E5 selecciona un documento del corpus. El chatbot recibe la pregunta y el contenido de ese documento para redactar una respuesta breve en español y citar la fuente. Evaluamos por separado si la respuesta explica la causa correcta, si conserva los hechos del documento y si incluye el identificador correspondiente.

In [ ]:
%%bash
set -e
cd /content/knowledge-hub-chatbot-base

PYTHONPATH=src .venv/bin/python - <<'PY'
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from kms_utils import load_documents
from semantic_search import MultilingualSearch

pregunta = "¿Por qué los pedidos se procesaron con tres horas de retraso?"

buscador = MultilingualSearch(load_documents())
fuente = buscador.search(pregunta, top_k=1).iloc[0]
del buscador

modelo_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
modelo = AutoModelForCausalLM.from_pretrained(modelo_id, dtype="auto")
modelo.eval()

mensajes = [
    {
        "role": "system",
        "content": (
            "Responde en español en una o dos frases usando únicamente la fuente. "
            "Explica lo que pregunta el usuario y cita el identificador entre corchetes. "
            "Si la fuente no permite responder, di que la información es insuficiente."
        ),
    },
    {
        "role": "user",
        "content": (
            f"Pregunta: {pregunta}\n\n"
            f"Fuente: [{fuente['id']}] {fuente['title']}\n{fuente['text']}"
        ),
    },
]

prompt = tokenizer.apply_chat_template(
    mensajes, tokenize=False, add_generation_prompt=True
)
entrada = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)

with torch.inference_mode():
    salida = modelo.generate(**entrada, max_new_tokens=120, do_sample=False)

respuesta = tokenizer.decode(
    salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True
).strip()

print("PREGUNTA:", pregunta)
print("\nRESPUESTA DEL CHATBOT:")
print(respuesta)
print(f"\nFUENTE RECUPERADA: [{fuente['id']}] {fuente['title']}")
print(fuente["text"])
print(f"\n¿Incluyó la cita esperada? {'Sí' if '[' + fuente['id'] + ']' in respuesta else 'No'}")
PY

PREGUNTA: ¿Por qué los pedidos se procesaron con tres horas de retraso?

RESPUESTA DEL CHATBOT:
La respuesta está dada por la fuente IR07, donde se explica que los pedidos se procesaron con tres horas de retraso debido a que las ordenes acumuladas en la cola fueron procesadas con un retraso de tres horas. El problema fue causado por un consumidor que se desintegró debido a un mensaje mal formado sin manejar correctamente. Una solución fue implementar un parche para resolver el problema y agregar monitoreo del nivel de cola para alertas adicionales.

FUENTE RECUPERADA: [IR07] Post-mortem: Message Queue Backlog
Order events accumulated in the queue and were processed with a delay of three hours. The root cause was a consumer that crashed due to an unhandled malformed message. A fix was deployed and monitoring of queue depth was added to the alerts.

¿Incluyó la cita esperada? No


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1074.21it/s]


### Ajuste de la primera respuesta

El buscador recuperó la fuente correcta. El chatbot identificó el mensaje malformado, pero tradujo de forma imprecisa la falla del proceso consumidor, repitió parte de la pregunta y agregó acciones correctivas que no se solicitaron. También mencionó IR07 sin el formato de cita esperado.

En la segunda prueba mantenemos el modelo, la pregunta y el buscador. Ajustamos la instrucción para obtener una explicación causal breve y dejamos que la aplicación adjunte el identificador de la fuente recuperada. Mostramos por separado el texto generado y la respuesta final para distinguir qué hizo el modelo y qué agregó el código.

In [ ]:
%%bash
set -e
cd /content/knowledge-hub-chatbot-base

PYTHONPATH=src .venv/bin/python - <<'PY'
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from kms_utils import load_documents
from semantic_search import MultilingualSearch

pregunta = "¿Por qué los pedidos se procesaron con tres horas de retraso?"

buscador = MultilingualSearch(load_documents())
fuente = buscador.search(pregunta, top_k=1).iloc[0]
del buscador

modelo_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
modelo = AutoModelForCausalLM.from_pretrained(modelo_id, dtype="auto")
modelo.eval()

mensajes = [
    {
        "role": "system",
        "content": (
            "Responde en español técnico natural usando solo el documento. "
            "Si preguntan por qué ocurrió algo, empieza por la causa y explica "
            "su consecuencia. Usa como máximo dos frases. Omite introducciones, "
            "identificadores y medidas correctivas salvo que te las pidan. "
            "Si el documento no contiene la respuesta, escribe exactamente: "
            "Información insuficiente."
        ),
    },
    {
        "role": "user",
        "content": f"Documento:\n{fuente['text']}\n\nPregunta:\n{pregunta}",
    },
]

prompt = tokenizer.apply_chat_template(
    mensajes, tokenize=False, add_generation_prompt=True
)
entrada = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)

with torch.inference_mode():
    salida = modelo.generate(**entrada, max_new_tokens=90, do_sample=False)

generado = tokenizer.decode(
    salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True
).strip()

if not generado or generado.casefold().rstrip(".") == "información insuficiente":
    respuesta = "Información insuficiente."
else:
    respuesta = f"{generado} [{fuente['id']}]"

print("PREGUNTA:", pregunta)
print("\nTEXTO GENERADO:", generado)
print("\nRESPUESTA FINAL:", respuesta)
print(f"\nFUENTE RECUPERADA: [{fuente['id']}] {fuente['title']}")
print(fuente["text"])
PY

PREGUNTA: ¿Por qué los pedidos se procesaron con tres horas de retraso?

TEXTO GENERADO: El problema fue causado por un consumidor que se desintegró debido a una mensajería mal formada sin manejar correctamente.

RESPUESTA FINAL: El problema fue causado por un consumidor que se desintegró debido a una mensajería mal formada sin manejar correctamente. [IR07]

FUENTE RECUPERADA: [IR07] Post-mortem: Message Queue Backlog
Order events accumulated in the queue and were processed with a delay of three hours. The root cause was a consumer that crashed due to an unhandled malformed message. A fix was deployed and monitoring of queue depth was added to the alerts.


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 4751.19it/s]


### Resultado del ajuste de instrucciones

La aplicación adjuntó correctamente la cita [IR07] y la respuesta fue más breve. Sin embargo, el modelo volvió a traducir de forma incorrecta la falla del proceso consumidor y omitió que los eventos se acumularon en la cola. Por ello, conservamos la cita automática y comparamos dos tamaños de modelo bajo las mismas condiciones.

In [ ]:
%%bash
set -e
cd /content/knowledge-hub-chatbot-base

PYTHONPATH=src .venv/bin/python - <<'PY'
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from kms_utils import load_documents

docs = load_documents().set_index("id")
casos = [
    ("IR07", "¿Por qué los pedidos se procesaron con tres horas de retraso?"),
    ("PP08", "¿Por dónde debo solicitar mis vacaciones y con cuánta anticipación?"),
]

instruccion = (
    "Responde en español técnico natural usando solo el documento. "
    "Si preguntan por qué ocurrió algo, empieza por la causa y explica "
    "su consecuencia. Usa como máximo dos frases. Omite introducciones, "
    "identificadores y medidas correctivas salvo que te las pidan. "
    "Si el documento no contiene la respuesta, escribe exactamente: "
    "Información insuficiente."
)

respuestas = {}
for modelo_id in ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]:
    tokenizer = AutoTokenizer.from_pretrained(modelo_id)
    modelo = AutoModelForCausalLM.from_pretrained(modelo_id, dtype="auto")
    modelo.eval()

    for doc_id, pregunta in casos:
        texto = docs.at[doc_id, "text"]
        mensajes = [
            {"role": "system", "content": instruccion},
            {"role": "user", "content": f"Documento:\n{texto}\n\nPregunta:\n{pregunta}"},
        ]
        prompt = tokenizer.apply_chat_template(
            mensajes, tokenize=False, add_generation_prompt=True
        )
        entrada = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)

        with torch.inference_mode():
            salida = modelo.generate(**entrada, max_new_tokens=90, do_sample=False)

        generado = tokenizer.decode(
            salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()
        respuestas[modelo_id, doc_id] = generado

    del modelo, tokenizer
    gc.collect()

for doc_id, pregunta in casos:
    print(f"\nPREGUNTA: {pregunta}")
    print(f"FUENTE [{doc_id}]: {docs.at[doc_id, 'text']}")
    for modelo_id in ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]:
        print(f"\n{modelo_id}: {respuestas[modelo_id, doc_id]}")
    print("-" * 70)
PY


PREGUNTA: ¿Por qué los pedidos se procesaron con tres horas de retraso?
FUENTE [IR07]: Order events accumulated in the queue and were processed with a delay of three hours. The root cause was a consumer that crashed due to an unhandled malformed message. A fix was deployed and monitoring of queue depth was added to the alerts.

Qwen/Qwen2.5-1.5B-Instruct: El problema fue causado por un consumidor que se desintegró debido a una mensajería mal formada sin manejar correctamente.

Qwen/Qwen2.5-3B-Instruct: Los pedidos se procesaron con tres horas de retraso porque una consumidora que se crashó debido a un mensaje mal formado causó un retraso. La solución fue implementada y se agregó monitoreo del nivel de la cola para los avisos.
----------------------------------------------------------------------

PREGUNTA: ¿Por dónde debo solicitar mis vacaciones y con cuánta anticipación?
FUENTE [PP08]: Employees are entitled to vacation days according to their seniority and local labor law. Leave re

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 622.02it/s]


### Prueba de información insuficiente con Qwen 3B

Usamos la política de vacaciones PP08 para hacer dos preguntas. El documento indica con cuánta anticipación se deben solicitar las vacaciones, pero no especifica cuántos días corresponden a una persona con cuatro años de antigüedad. Evaluamos si el chatbot recupera el plazo correcto y reconoce cuándo falta el dato necesario para responder.

In [ ]:
%%bash
set -e
cd /content/knowledge-hub-chatbot-base

PYTHONPATH=src .venv/bin/python - <<'PY'
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from kms_utils import load_documents

fuente = load_documents().set_index("id").at["PP08", "text"]
modelo_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_id)
modelo = AutoModelForCausalLM.from_pretrained(modelo_id, dtype="auto").eval()

preguntas = [
    "¿Con cuánta anticipación debo solicitar mis vacaciones?",
    "Si llevo exactamente cuatro años en la empresa, ¿cuántos días de vacaciones me corresponden?",
]

print("MODELO:", modelo_id)
print("FUENTE [PP08]:", fuente)

for pregunta in preguntas:
    mensajes = [
        {
            "role": "system",
            "content": (
                "Responde brevemente en español usando únicamente el documento. "
                "Si falta un dato necesario para responder, escribe: "
                "Información insuficiente."
            ),
        },
        {
            "role": "user",
            "content": f"Documento:\n{fuente}\n\nPregunta:\n{pregunta}",
        },
    ]
    prompt = tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    entrada = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)

    with torch.inference_mode():
        salida = modelo.generate(**entrada, max_new_tokens=60, do_sample=False)

    respuesta = tokenizer.decode(
        salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    print(f"\nPREGUNTA: {pregunta}")
    print(f"RESPUESTA: {respuesta}")
PY

MODELO: Qwen/Qwen2.5-3B-Instruct
FUENTE [PP08]: Employees are entitled to vacation days according to their seniority and local labor law. Leave requests must be submitted through the HR portal at least two weeks in advance. Managers approve requests considering project deadlines and team coverage.

PREGUNTA: ¿Con cuánta anticipación debo solicitar mis vacaciones?
RESPUESTA: Debes solicitar tus vacaciones al menos dos semanas antes.

PREGUNTA: Si llevo exactamente cuatro años en la empresa, ¿cuántos días de vacaciones me corresponden?
RESPUESTA: Información insuficiente.


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 3208.10it/s]


### Resultado de la prueba de información insuficiente

Qwen 3B respondió correctamente las dos preguntas sobre PP08. Conservó el plazo mínimo de dos semanas para solicitar vacaciones y reconoció que el documento no permite calcular cuántos días corresponden a una persona con cuatro años de antigüedad.

En esta prueba entregamos PP08 directamente al modelo. El siguiente paso es comprobar el flujo completo: que E5 seleccione la fuente a partir de la pregunta y que el chatbot responda o se abstenga usando el documento recuperado.

In [ ]:
%%bash
set -e
cd /content/knowledge-hub-chatbot-base

PYTHONPATH=src .venv/bin/python - <<'PY'
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from kms_utils import load_documents
from semantic_search import MultilingualSearch

buscador = MultilingualSearch(load_documents())
modelo_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
modelo = AutoModelForCausalLM.from_pretrained(modelo_id, dtype="auto").eval()

def responder(pregunta):
    fuente = buscador.search(pregunta, top_k=1).iloc[0]
    mensajes = [
        {"role": "system", "content": (
            "Responde brevemente en español usando únicamente el documento. "
            "Si falta un dato necesario para responder, escribe exactamente: "
            "Información insuficiente."
        )},
        {"role": "user", "content": f"Documento:\n{fuente['text']}\n\nPregunta:\n{pregunta}"},
    ]
    prompt = tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    entrada = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    with torch.inference_mode():
        salida = modelo.generate(**entrada, max_new_tokens=60, do_sample=False)

    texto = tokenizer.decode(
        salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    if not texto or texto.casefold().rstrip(".") == "información insuficiente":
        respuesta = "Información insuficiente."
    else:
        respuesta = f"{texto} [{fuente['id']}]"

    print(f"\nPregunta: {pregunta}")
    print(f"Fuente consultada: [{fuente['id']}] {fuente['title']}")
    print(f"Respuesta: {respuesta}")

responder("¿Con cuánta anticipación debo solicitar mis vacaciones?")
responder("Si llevo exactamente cuatro años en la empresa, ¿cuántos días de vacaciones me corresponden?")
PY


Pregunta: ¿Con cuánta anticipación debo solicitar mis vacaciones?
Fuente consultada: [PP08] Vacation and Leave Policy
Respuesta: Debes solicitar tus vacaciones al menos dos semanas antes. [PP08]

Pregunta: Si llevo exactamente cuatro años en la empresa, ¿cuántos días de vacaciones me corresponden?
Fuente consultada: [PP08] Vacation and Leave Policy
Respuesta: Información insuficiente.


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 4383.31it/s]


### Verificación del chatbot integrado

Ejecutamos el módulo `src/chatbot.py` desde la rama del proyecto y registramos el commit evaluado. Usamos dos preguntas sobre vacaciones para comprobar si el módulo cita PP08 cuando encuentra el plazo de solicitud y reconoce la falta de información cuando se pide una cantidad de días que la política no especifica.

In [ ]:
%%bash
set -euo pipefail
cd /content/knowledge-hub-chatbot-base

git fetch --quiet origin main
git switch --quiet --detach FETCH_HEAD
printf 'Versión evaluada: '
git rev-parse --short HEAD

.venv/bin/python src/chatbot.py \
  "¿Con cuánta anticipación debo solicitar mis vacaciones?" \
  "Si llevo exactamente cuatro años en la empresa, ¿cuántos días de vacaciones me corresponden?"

Versión evaluada: 0c568a5

Pregunta: ¿Con cuánta anticipación debo solicitar mis vacaciones?
Fuente consultada: [PP08] Vacation and Leave Policy
Respuesta: Debes solicitar tus vacaciones al menos dos semanas antes. [PP08]

Pregunta: Si llevo exactamente cuatro años en la empresa, ¿cuántos días de vacaciones me corresponden?
Fuente consultada: [PP08] Vacation and Leave Policy
Respuesta: Información insuficiente.


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 1841.30it/s]
